# MDR-TS v23.1
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v23.1
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_8.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

- Implementing other models (SVR, SVM, KNN, GB, DT)

## 0. Imports

In [5]:
import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from xgboost import XGBRegressor

import torch

project_root = os.path.abspath("../../")
if project_root not in sys.path:
    sys.path.append(project_root)

from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

models_root = os.path.abspath("../../")
if models_root not in sys.path:
    sys.path.insert(0, models_root)

# from Temporal.Utils.dashboard import metrics_dashboard

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

imports loaded
using: cpu


In [6]:
SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: False
environment setup complete


In [7]:
VERSION = "v23"
SUBVERSION = "v23.1"
RUN_NAME = "mdr_ts_v23_1"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v23/v23.1

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [8]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/train.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/val.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_8.0/test.csv


In [9]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 499

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'elev', 'slope', 'aspect', 'DOY', 'SMAP_sm_am_interp', 'SMAP_sm_pm_interp', 'soil_moisture_5cm', 'J_aspect_deg', 'J_bio_bio01', 'J_bio_bio02', 'J_bio_bio03', 'J_bio_bio04', 'J_bio_bio05', 'J_bio_bio06', 'J_bio_bio07', 'J_bio_bio08', 'J_bio_bio09', 'J_bio_bio10']


In [10]:
TARGET_COL = "soil_moisture_5cm"

KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_ema_G_API_kobs7',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0'
    ]

expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

Columns locked
  Features: 38
  Target:   soil_moisture_5cm


In [11]:
corr = train_df[FEATURE_COLS].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# find pairs > 0.995 correlation
high_corr = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.995
]

print("Highly correlated pairs:")
for a, b, c in high_corr:
    print(f"{a} <-> {b} : {c:.5f}")

Highly correlated pairs:
V_rollmean_G_API_kobs7 <-> V_ema_G_API_kobs7 : 0.99727


In [12]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics):
    print(f"{'METRIC':<15} | {'VALUE':<10}")
    print("-" * 28)
    for k, v in metrics.items():
        val_str = f"{v:,}" if isinstance(v, int) else f"{v:+.5f}"
        print(f"{k:<15} | {val_str:<10}")

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [13]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     6868
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2017-01-01 -- 2020-12-31

VAL
  rows:     2720
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-01-01 -- 2022-12-31

TEST
  rows:     4016
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2023-01-01 -- 2025-12-31

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


In [14]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

trainval_df_d["date"] = pd.to_datetime(trainval_df_d["date"], errors="coerce")
trainval_df_d["year"] = trainval_df_d["date"].dt.year.astype(float)

max_year = trainval_df_d["year"].max()
beta = 0.2

w_trainval = np.exp(beta * (trainval_df_d["year"] - max_year))
w_trainval = w_trainval / w_trainval.mean()

In [15]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_trainval_d = trainval_df_d[FEATURE_COLS].copy()
y_trainval_d = trainval_df_d[TARGET_COL].copy()

X_test_d = test_df[FEATURE_COLS].copy()
y_test_d = test_df[TARGET_COL].copy()

print("\nDRIFT matrices:")
print("  X_trainval_d:", X_trainval_d.shape)
print("  X_test_d:    ", X_test_d.shape)


DRIFT matrices:
  X_trainval_d: (9588, 38)
  X_test_d:     (4016, 38)


---
### Base Model

In [16]:
XGB_PARAMS_DRIFT_W = dict(
    objective="reg:pseudohubererror",
    random_state=SEED,
    n_jobs=-1,
    subsample=0.9,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=2,
    n_estimators=5500,
    learning_rate=0.04,
    reg_lambda=1.5,
    reg_alpha=0.03,
    gamma=0.0,
)

In [17]:
w_trainval_s = pd.Series(w_trainval)
years_tv = trainval_df_d["year"].reset_index(drop=True)

print("Weight stats:")
print(w_trainval_s.describe())

min_year = years_tv.min()
max_year = years_tv.max()

print("Min year weight:", float(w_trainval_s[years_tv == min_year].mean()))
print("Max year weight:", float(w_trainval_s[years_tv == max_year].mean()))

Weight stats:
count    9588.000000
mean        1.000000
std         0.336108
min         0.591030
25%         0.721885
50%         0.881713
75%         1.315361
max         1.606585
Name: year, dtype: float64
Min year weight: 0.5910295759572686
Max year weight: 1.6065849564064987


In [18]:
trainval_df_d = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

xgb_drift_w_tv = XGBRegressor(**XGB_PARAMS_DRIFT_W)
xgb_drift_w_tv.fit(
  trainval_df_d[FEATURE_COLS],
  trainval_df_d[TARGET_COL],
  verbose=0,
  sample_weight=w_trainval
  )

y_test_d_w = np.asarray(test_df[TARGET_COL]).ravel()
pred_drift_test_w = np.asarray(xgb_drift_w_tv.predict(test_df[FEATURE_COLS])).ravel()

In [19]:
print("======= WEIGHTED MODEL METRICS =======")
neat_print(get_metrics_dict(y_test_d, pred_drift_test_w, prefix="no_w_"))

======= WEIGHTED MODEL METRICS =======
METRIC          | VALUE     
----------------------------
no_w_n          | 4,016     
no_w_r2         | +0.82239  
no_w_mae        | +0.02832  
no_w_rmse       | +0.03968  
no_w_ubrmse     | +0.03956  
no_w_bias       | -0.00320  
no_w_med_ae     | +0.02032  
no_w_p90_ae     | +0.06152  
no_w_q05_err    | -0.06826  
no_w_q50_err    | -0.00285  
no_w_q95_err    | +0.05462  


---

### SVM (Support Vector Machine)

In [20]:
mask = X_trainval_d.notna().all(axis=1) & y_trainval_d.notna()
X_train_clean = X_trainval_d[mask].copy()
y_train_clean = y_trainval_d[mask].copy()
w_train_clean = w_trainval[mask].copy()

mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

svm_drift_w_tv = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1))
])

svm_drift_w_tv.fit(X_train_clean, y_train_clean, model__sample_weight=w_train_clean)
pred_svm_drift_test_w = svm_drift_w_tv.predict(X_test_clean)

neat_print(get_metrics_dict(y_test_clean, pred_svm_drift_test_w, prefix="svm_w_"))

METRIC          | VALUE     
----------------------------
svm_w_n         | 3,846     
svm_w_r2        | +0.58051  
svm_w_mae       | +0.04984  
svm_w_rmse      | +0.06216  
svm_w_ubrmse    | +0.06051  
svm_w_bias      | -0.01424  
svm_w_med_ae    | +0.04296  
svm_w_p90_ae    | +0.10598  
svm_w_q05_err   | -0.12128  
svm_w_q50_err   | -0.00646  
svm_w_q95_err   | +0.07575  


---
### SVR (Support Vector Regression)

In [21]:
mask = X_trainval_d.notna().all(axis=1) & y_trainval_d.notna()
X_train_clean = X_trainval_d[mask].copy()
y_train_clean = y_trainval_d[mask].copy()
w_train_clean = w_trainval[mask].copy()

mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

svr_drift_w_tv = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.05))
])

svr_drift_w_tv.fit(X_train_clean, y_train_clean, model__sample_weight=w_train_clean)
pred_svr_drift_test_w = svr_drift_w_tv.predict(X_test_clean)

neat_print(get_metrics_dict(y_test_clean, pred_svr_drift_test_w, prefix="svr_w_"))

METRIC          | VALUE     
----------------------------
svr_w_n         | 3,846     
svr_w_r2        | +0.68561  
svr_w_mae       | +0.04177  
svr_w_rmse      | +0.05382  
svr_w_ubrmse    | +0.05230  
svr_w_bias      | -0.01268  
svr_w_med_ae    | +0.03250  
svr_w_p90_ae    | +0.09243  
svr_w_q05_err   | -0.10775  
svr_w_q50_err   | -0.00928  
svr_w_q95_err   | +0.07403  


---
### KNN (K-nearest neighbors) | No weights

In [22]:
mask = X_trainval_d.notna().all(axis=1) & y_trainval_d.notna()
X_train_clean = X_trainval_d[mask].copy()
y_train_clean = y_trainval_d[mask].copy()
w_train_clean = w_trainval[mask].copy()

mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

knn_drift_w_tv = Pipeline([
    ('scaler', StandardScaler()),
    ('model', KNeighborsRegressor(n_neighbors=5, weights='distance'))
])

knn_drift_w_tv.fit(X_train_clean, y_train_clean)
pred_knn_drift_test_w = knn_drift_w_tv.predict(X_test_clean)

neat_print(get_metrics_dict(y_test_clean, pred_knn_drift_test_w, prefix="knn_w_"))

METRIC          | VALUE     
----------------------------
knn_w_n         | 3,846     
knn_w_r2        | +0.64868  
knn_w_mae       | +0.04061  
knn_w_rmse      | +0.05689  
knn_w_ubrmse    | +0.05672  
knn_w_bias      | -0.00435  
knn_w_med_ae    | +0.02813  
knn_w_p90_ae    | +0.09764  
knn_w_q05_err   | -0.10460  
knn_w_q50_err   | -0.00326  
knn_w_q95_err   | +0.08936  


---
### GB (Gradient Boosting)

In [23]:
mask = X_trainval_d.notna().all(axis=1) & y_trainval_d.notna()
X_train_clean = X_trainval_d[mask].copy()
y_train_clean = y_trainval_d[mask].copy()
w_train_clean = w_trainval[mask].copy()

mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

gb_drift_w_tv = GradientBoostingRegressor(
    random_state=SEED,
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    min_samples_leaf=8,
    subsample=0.9,
)

gb_drift_w_tv.fit(X_train_clean, y_train_clean, sample_weight=w_train_clean)
pred_gb_drift_test_w = gb_drift_w_tv.predict(X_test_clean)

neat_print(get_metrics_dict(y_test_clean, pred_gb_drift_test_w, prefix="gb_w_"))

METRIC          | VALUE     
----------------------------
gb_w_n          | 3,846     
gb_w_r2         | +0.80083  
gb_w_mae        | +0.03149  
gb_w_rmse       | +0.04283  
gb_w_ubrmse     | +0.04231  
gb_w_bias       | -0.00666  
gb_w_med_ae     | +0.02379  
gb_w_p90_ae     | +0.06848  
gb_w_q05_err    | -0.07777  
gb_w_q50_err    | -0.00680  
gb_w_q95_err    | +0.05959  


---
### DT (Decision Tree)

In [24]:
mask = X_trainval_d.notna().all(axis=1) & y_trainval_d.notna()
X_train_clean = X_trainval_d[mask].copy()
y_train_clean = y_trainval_d[mask].copy()
w_train_clean = w_trainval[mask].copy()

mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

dt_drift_w_tv = DecisionTreeRegressor(
    random_state=SEED,
    max_depth=12,
    min_samples_leaf=10,
)

dt_drift_w_tv.fit(X_train_clean, y_train_clean, sample_weight=w_train_clean)
pred_dt_drift_test_w = dt_drift_w_tv.predict(X_test_clean)

neat_print(get_metrics_dict(y_test_clean, pred_dt_drift_test_w, prefix="dt_w_"))

METRIC          | VALUE     
----------------------------
dt_w_n          | 3,846     
dt_w_r2         | +0.69773  
dt_w_mae        | +0.03736  
dt_w_rmse       | +0.05277  
dt_w_ubrmse     | +0.05275  
dt_w_bias       | -0.00153  
dt_w_med_ae     | +0.02503  
dt_w_p90_ae     | +0.09096  
dt_w_q05_err    | -0.09895  
dt_w_q50_err    | -0.00004  
dt_w_q95_err    | +0.08600  


---


In [25]:
mask_test = X_test_d.notna().all(axis=1) & y_test_d.notna()
X_test_clean = X_test_d[mask_test].copy()
y_test_clean = y_test_d[mask_test].copy()

all_preds = {
    'XGB (BASE)': pred_drift_test_w[mask_test],
    'SVM': pred_svm_drift_test_w,
    'SVR': pred_svr_drift_test_w,
    'KNN': pred_knn_drift_test_w,
    'GB': pred_gb_drift_test_w,
    'DT': pred_dt_drift_test_w,
}

comparison_results = {}
for model_name, preds in all_preds.items():
    metrics = get_metrics_dict(y_test_clean, preds, prefix="")
    comparison_results[model_name] = metrics

comp_df = pd.DataFrame(comparison_results).T
comp_df = comp_df[['n', 'r2', 'mae', 'ubrmse', 'bias']]

print("\nMETRICS SUMMARY:")
print(comp_df.to_string())


METRICS SUMMARY:
                 n        r2       mae    ubrmse      bias
XGB (BASE)  3846.0  0.825629  0.028746  0.039956 -0.003137
SVM         3846.0  0.580511  0.049843  0.060511 -0.014240
SVR         3846.0  0.685606  0.041767  0.052301 -0.012682
KNN         3846.0  0.648681  0.040611  0.056722 -0.004348
GB          3846.0  0.800830  0.031487  0.042312 -0.006664
DT          3846.0  0.697734  0.037363  0.052746 -0.001535
